# PCA and UMAP for Embeddings Colored by Model Evaluation Metrics

This notebook extends the embedding PCA/UMAP workflow to overlay per-model evaluation metrics from CoNIC/Lizard evaluation CSVs.

Intended use:
- load the expanded embedding table with metadata and vectors
- detect embedding columns automatically from a prefix
- reuse the same PCA/UMAP settings as the base embedding notebook
- load one evaluation CSV per model from `outputs/conic_liz`
- map each evaluation row back to the embedding table using `image_id -> sample_id`
- convert `pixel_dice` and `instance_pq` into `low` / `mid` / `high` categorical overlays using quantile cutoffs
- generate PCA and UMAP plots for each model/metric combination to inspect whether high-performing patches cluster

This notebook assumes the evaluation CSVs follow the existing `outputs/conic_liz/*_evaluation.csv` pattern and that `image_id` resolves back to the embedding table sample ID.


## Imports

In [ ]:
import socket, torch
print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    import umap
except ImportError as exc:
    raise ImportError(
        "This notebook requires umap-learn. Install it in the environment before running the UMAP cells."
    ) from exc

plt.rcParams["figure.dpi"] = 120


## User Configuration

In [ ]:
REPO_ROOT = None  # Optional: set this if you open the notebook from outside the repo root.
INPUT_EMBEDDING_CSV = Path("outputs/conic_liz/embedding_tables/embed_morph_with_vectors.csv")
EVALUATION_CSV_GLOB = "*_evaluation.csv"
EVALUATION_DIR = Path("outputs/conic_liz")
EMBEDDING_PREFIX = "emb_"

PCA_N_COMPONENTS = 400
UMAP_INPUT_PCA_COMPONENTS = 400
STANDARDIZE_EMBEDDINGS = False  # Learned embeddings often behave better without per-dimension z-scoring.
SHUFFLE_ROWS_BEFORE_EMBEDDING = True
SHUFFLE_PLOT_ORDER = True

UMAP_N_NEIGHBORS = 75
UMAP_MIN_DIST = 0.3
UMAP_METRIC = "euclidean"
UMAP_INIT = "spectral"
RANDOM_SEED = 7

FIGSIZE = (8, 6)
POINT_SIZE = 11
ALPHA = 0.75

MODEL_METRIC_COLUMNS = ["pixel_dice", "instance_pq"]
METRIC_CATEGORY_SUFFIX = "_level"
LOW_QUANTILE = 0.25
HIGH_QUANTILE = 0.75
CATEGORY_ORDER = ["low", "mid", "high", "Missing"]
PRIMARY_MODEL_NAME = None  # Set to a model name like "cellvit_sam" for a single-model preview.
PRIMARY_METRIC = "instance_pq"
ONLY_STATUS = "ok"

RUN_KMEANS = False
KMEANS_N_CLUSTERS = 8
KMEANS_LABEL_COLUMN = "kmeans_cluster"

PLOT_OUTPUT_DIR = Path("outputs/conic_liz/embedding_tables/model_metric_plots")


## Load Embedding Data


In [ ]:
def find_repo_root(start: Path) -> Path | None:
    start_path = start.expanduser().resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / ".git").exists():
            return candidate
    return None


def resolve_project_path(path_like: Path, repo_root: Path | None = None) -> Path:
    path = Path(path_like).expanduser()
    if path.is_absolute():
        return path.resolve()

    candidates = []
    if repo_root is not None:
        candidates.append((repo_root / path).resolve())
    candidates.append(path.resolve())

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


repo_root = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT is not None else find_repo_root(Path.cwd())
input_embedding_csv = resolve_project_path(INPUT_EMBEDDING_CSV, repo_root=repo_root)
evaluation_dir = resolve_project_path(EVALUATION_DIR, repo_root=repo_root)
plot_output_dir = resolve_project_path(PLOT_OUTPUT_DIR, repo_root=repo_root)

if not input_embedding_csv.exists():
    raise FileNotFoundError(
        f"Embedding CSV does not exist: {input_embedding_csv}. Update INPUT_EMBEDDING_CSV in the config cell before running further."
    )
if not evaluation_dir.exists():
    raise FileNotFoundError(
        f"Evaluation directory does not exist: {evaluation_dir}. Update EVALUATION_DIR in the config cell before running further."
    )

if repo_root is not None:
    print(f"Resolved repo root: {repo_root}")
print(f"Resolved embedding CSV: {input_embedding_csv}")
print(f"Resolved evaluation directory: {evaluation_dir}")
print(f"Plot output directory for optional saves: {plot_output_dir}")

embedding_df = pd.read_csv(input_embedding_csv)
print(f"Loaded {len(embedding_df):,} rows x {len(embedding_df.columns):,} columns from {input_embedding_csv}")
embedding_df.head()


## Detect Embedding Columns


In [ ]:
def embedding_sort_key(column_name: str):
    suffix = str(column_name)[len(EMBEDDING_PREFIX):]
    return (0, int(suffix)) if suffix.isdigit() else (1, str(column_name))


embedding_cols = [column for column in embedding_df.columns if str(column).startswith(EMBEDDING_PREFIX)]
if not embedding_cols:
    raise ValueError(
        f"No embedding columns were found with prefix {EMBEDDING_PREFIX!r}. "
        "Update EMBEDDING_PREFIX or check the input CSV."
    )

embedding_cols = sorted(embedding_cols, key=embedding_sort_key)
metadata_cols = [column for column in embedding_df.columns if column not in embedding_cols]

print(f"Detected {len(embedding_cols):,} embedding columns.")
print(f"First embedding columns: {embedding_cols[:5]}")
print(f"Detected {len(metadata_cols):,} non-embedding columns.")
print(f"Sample join columns available: {[column for column in ['sample_id', 'image_path', 'mask_path'] if column in embedding_df.columns]}")


## Build Embedding Matrix


In [ ]:
if SHUFFLE_ROWS_BEFORE_EMBEDDING:
    working_df = embedding_df.sample(frac=1.0, random_state=RANDOM_SEED).copy()
    print("Shuffled dataframe rows before PCA/UMAP fitting to remove row-order dependence.")
else:
    working_df = embedding_df.copy()
    print("Using the original dataframe row order for PCA/UMAP fitting.")

X = working_df[embedding_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
if X.ndim != 2 or X.shape[1] == 0:
    raise ValueError(f"Expected a 2D embedding matrix with at least one column, got shape {X.shape}.")

non_finite_mask = ~np.isfinite(X)
if non_finite_mask.any():
    raise ValueError(
        f"Embedding matrix contains {non_finite_mask.sum():,} non-finite values. "
        "Check the exported embedding CSV before running PCA/UMAP."
    )

if X.shape[0] < 2:
    raise ValueError("At least two rows are required for PCA/UMAP exploration.")

print(f"Embedding matrix shape: {X.shape}")


## Optional Standardization


In [ ]:
if STANDARDIZE_EMBEDDINGS:
    scaler = StandardScaler()
    X_model = scaler.fit_transform(X)
    print("Applied StandardScaler to embedding columns before PCA.")
else:
    scaler = None
    X_model = X.copy()
    print("Using raw embeddings without standardization.")


## PCA


In [ ]:
max_pca_components = min(PCA_N_COMPONENTS, X_model.shape[0], X_model.shape[1])
if max_pca_components < 2:
    raise ValueError(
        f"Need at least 2 PCA components for plotting, but only {max_pca_components} component(s) are possible."
    )
if max_pca_components != PCA_N_COMPONENTS:
    print(f"Adjusted PCA components from {PCA_N_COMPONENTS} to {max_pca_components} based on data shape.")

pca = PCA(n_components=max_pca_components, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_model)
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print(f"PCA output shape: {X_pca.shape}")
print(f"First five explained variance ratios: {explained_variance_ratio[:5]}")
print(f"Cumulative explained variance at component {len(cumulative_explained_variance)}: {cumulative_explained_variance[-1]:.4f}")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(np.arange(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker="o", linewidth=1.5)
ax.set_xlabel("PCA component")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA cumulative explained variance")
ax.grid(alpha=0.3)
plt.tight_layout()
# plt.savefig(plot_output_dir / "pca_cumulative_explained_variance.png", bbox_inches="tight")
plt.show()


## UMAP

In [ ]:
umap_input_components = min(UMAP_INPUT_PCA_COMPONENTS, X_pca.shape[1])
if umap_input_components < 2:
    raise ValueError(
        f"Need at least 2 PCA components for UMAP, but only {umap_input_components} component(s) are available."
    )
if umap_input_components != UMAP_INPUT_PCA_COMPONENTS:
    print(
        f"Adjusted UMAP input PCA components from {UMAP_INPUT_PCA_COMPONENTS} to {umap_input_components} based on data shape."
    )

X_umap_input = X_pca[:, :umap_input_components]

umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    init=UMAP_INIT,
    random_state=RANDOM_SEED,
)
X_umap = umap_model.fit_transform(X_umap_input)
print(f"UMAP input shape: {X_umap_input.shape}")
print(f"UMAP output shape: {X_umap.shape}")


## Load and Map Evaluation Metrics


In [ ]:
def make_quantile_category(
    series: pd.Series,
    *,
    low_quantile: float = LOW_QUANTILE,
    high_quantile: float = HIGH_QUANTILE,
    suffix: str = METRIC_CATEGORY_SUFFIX,
):
    numeric_values = pd.to_numeric(series, errors="coerce")
    valid_values = numeric_values.dropna()

    categorized = pd.Series("Missing", index=series.index, dtype="object")
    if valid_values.empty:
        summary = {
            "low_threshold": np.nan,
            "high_threshold": np.nan,
            "note": "all values missing",
        }
    else:
        low_threshold = valid_values.quantile(low_quantile)
        high_threshold = valid_values.quantile(high_quantile)

        if np.isclose(low_threshold, high_threshold):
            categorized.loc[numeric_values.notna()] = "mid"
            summary = {
                "low_threshold": float(low_threshold),
                "high_threshold": float(high_threshold),
                "note": "quantile thresholds collapsed; assigned all non-missing values to mid",
            }
        else:
            categorized.loc[numeric_values.notna()] = "mid"
            categorized.loc[numeric_values <= low_threshold] = "low"
            categorized.loc[numeric_values >= high_threshold] = "high"
            summary = {
                "low_threshold": float(low_threshold),
                "high_threshold": float(high_threshold),
                "note": "",
            }

    categorized = pd.Series(
        pd.Categorical(categorized, categories=CATEGORY_ORDER, ordered=True),
        index=series.index,
        name=f"{series.name}{suffix}",
    )
    return categorized, summary


def derive_sample_id_from_image_id(image_id: pd.Series) -> pd.Series:
    return (
        image_id.astype(str)
        .str.split("/")
        .str[-1]
        .str.replace(r"_image\.[^.]+$", "", regex=True)
    )


projection_df = pd.DataFrame(
    {
        "pca_1": X_pca[:, 0],
        "pca_2": X_pca[:, 1],
        "umap_1": X_umap[:, 0],
        "umap_2": X_umap[:, 1],
    },
    index=working_df.index,
)
plot_df = embedding_df.join(projection_df)

if "sample_id" not in plot_df.columns:
    raise KeyError("The embedding table must contain 'sample_id' to map evaluation rows back onto the embeddings.")

evaluation_paths = sorted(evaluation_dir.glob(EVALUATION_CSV_GLOB))
if not evaluation_paths:
    raise FileNotFoundError(
        f"No evaluation CSVs matched {EVALUATION_CSV_GLOB!r} under {evaluation_dir}."
    )

model_metric_plot_columns = []
quantile_summary_rows = []
join_summary_rows = []

for evaluation_path in evaluation_paths:
    evaluation_df = pd.read_csv(evaluation_path)
    if ONLY_STATUS is not None and "status" in evaluation_df.columns:
        evaluation_df = evaluation_df.loc[evaluation_df["status"] == ONLY_STATUS].copy()

    required_columns = {"image_id", *MODEL_METRIC_COLUMNS}
    missing_columns = sorted(required_columns.difference(evaluation_df.columns))
    if missing_columns:
        raise KeyError(f"{evaluation_path.name} is missing required columns: {missing_columns}")

    if "model_name" in evaluation_df.columns and evaluation_df["model_name"].dropna().nunique() == 1:
        model_name = str(evaluation_df["model_name"].dropna().iloc[0])
    else:
        model_name = evaluation_path.name.removesuffix("_evaluation.csv")

    evaluation_df = evaluation_df.copy()
    evaluation_df["sample_id"] = derive_sample_id_from_image_id(evaluation_df["image_id"])
    evaluation_df = evaluation_df.drop_duplicates(subset=["sample_id"], keep="last")

    metric_columns = ["sample_id", *MODEL_METRIC_COLUMNS]
    renamed_columns = {
        metric_column: f"{model_name}__{metric_column}"
        for metric_column in MODEL_METRIC_COLUMNS
    }
    model_metrics_df = evaluation_df[metric_columns].rename(columns=renamed_columns)
    plot_df = plot_df.merge(model_metrics_df, on="sample_id", how="left", validate="one_to_one")

    matched_rows = plot_df[f"{model_name}__{MODEL_METRIC_COLUMNS[0]}"].notna().sum()
    join_summary_rows.append(
        {
            "model_name": model_name,
            "evaluation_rows_after_filter": len(evaluation_df),
            "matched_embedding_rows": int(matched_rows),
            "embedding_rows_total": len(plot_df),
        }
    )

    for metric_column in MODEL_METRIC_COLUMNS:
        raw_column = f"{model_name}__{metric_column}"
        level_column = f"{raw_column}{METRIC_CATEGORY_SUFFIX}"
        categorized_series, summary = make_quantile_category(plot_df[raw_column])
        plot_df[level_column] = categorized_series
        model_metric_plot_columns.append((model_name, metric_column, raw_column, level_column))
        quantile_summary_rows.append(
            {
                "model_name": model_name,
                "metric": metric_column,
                **summary,
            }
        )

join_summary_df = pd.DataFrame(join_summary_rows)
quantile_summary_df = pd.DataFrame(quantile_summary_rows)

print("Loaded evaluation CSVs:")
print([path.name for path in evaluation_paths])
print("\nJoin summary:")
print(join_summary_df)
print("\nQuantile thresholds used for model metrics:")
print(quantile_summary_df)

preview_metric_columns = [level_column for _, _, _, level_column in model_metric_plot_columns[:4]]
plot_df[["sample_id", "pca_1", "pca_2", "umap_1", "umap_2", *preview_metric_columns]].head()


## Plotting Helpers


In [ ]:
def ensure_columns_present(frame: pd.DataFrame, columns, *, context: str = ""):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        prefix = f"{context}: " if context else ""
        raise KeyError(f"{prefix}missing columns: {missing}")


def plot_embedding_scatter(
    frame: pd.DataFrame,
    x_col: str,
    y_col: str,
    color_col: str | None = None,
    *,
    ax=None,
    title: str | None = None,
    figsize=FIGSIZE,
    point_size: float = POINT_SIZE,
    alpha: float = ALPHA,
    max_legend_items: int = 25,
):
    ensure_columns_present(frame, [x_col, y_col], context="plot_embedding_scatter")
    subset = frame.loc[frame[x_col].notna() & frame[y_col].notna()].copy()
    if subset.empty:
        raise ValueError(f"No rows with non-null coordinates for {x_col!r} and {y_col!r}.")

    if SHUFFLE_PLOT_ORDER and len(subset) > 1:
        subset = subset.sample(frac=1.0, random_state=RANDOM_SEED).copy()

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    if color_col is None:
        ax.scatter(subset[x_col], subset[y_col], s=point_size, alpha=alpha, c="steelblue", linewidths=0)
    else:
        ensure_columns_present(subset, [color_col], context="plot_embedding_scatter")
        color_series = subset[color_col]
        categorical_values = color_series.astype("object").where(color_series.notna(), "Missing").astype(str)
        if categorical_values.nunique() > max_legend_items:
            top_values = categorical_values.value_counts().nlargest(max_legend_items - 1).index.tolist()
            categorical_values = categorical_values.where(categorical_values.isin(top_values), "Other")

        present_values = set(categorical_values.unique())
        if isinstance(color_series.dtype, pd.CategoricalDtype):
            categories = [category for category in color_series.cat.categories if str(category) in present_values]
        else:
            categories = list(pd.Index(categorical_values.unique()))

        level_color_lookup = {
            "low": "#3b82f6",
            "mid": "#9ca3af",
            "high": "#ef4444",
            "Missing": "#d1d5db",
        }
        if set(categories).issubset(set(CATEGORY_ORDER)):
            categories = [category for category in CATEGORY_ORDER if category in categories]
            color_lookup = {category: level_color_lookup[category] for category in categories}
        else:
            cmap_name = "tab20" if len(categories) <= 20 else "gist_ncar"
            cmap = plt.get_cmap(cmap_name, len(categories))
            color_lookup = {category: cmap(index) for index, category in enumerate(categories)}

        point_colors = [color_lookup[value] for value in categorical_values]
        ax.scatter(subset[x_col], subset[y_col], s=point_size, alpha=alpha, c=point_colors, linewidths=0)
        handles = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=color_lookup[category],
                label=str(category),
                markersize=6,
            )
            for category in categories
        ]
        ax.legend(handles=handles, title=color_col, bbox_to_anchor=(1.02, 1), loc="upper left")

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title or f"{y_col} vs {x_col}")
    ax.grid(alpha=0.25)
    return ax


def plot_pca_and_umap(frame: pd.DataFrame, color_col: str | None = None, *, title_prefix: str | None = None):
    color_label = color_col if color_col is not None else "no color overlay"
    prefix = f"{title_prefix} | " if title_prefix else ""
    fig, axes = plt.subplots(1, 2, figsize=(FIGSIZE[0] * 2, FIGSIZE[1]))
    plot_embedding_scatter(frame, "pca_1", "pca_2", color_col=color_col, ax=axes[0], title=f"{prefix}PCA colored by {color_label}")
    plot_embedding_scatter(frame, "umap_1", "umap_2", color_col=color_col, ax=axes[1], title=f"{prefix}UMAP colored by {color_label}")
    plt.tight_layout()
    return fig, axes


## Single Model Preview


In [ ]:
if PRIMARY_MODEL_NAME is None:
    print("Set PRIMARY_MODEL_NAME in the config cell to render a single-model preview plot.")
else:
    preview_level_column = f"{PRIMARY_MODEL_NAME}__{PRIMARY_METRIC}{METRIC_CATEGORY_SUFFIX}"
    if preview_level_column not in plot_df.columns:
        available_preview_columns = [level_column for _, _, _, level_column in model_metric_plot_columns]
        raise KeyError(
            f"Preview column {preview_level_column!r} is not present. Available level columns: {available_preview_columns}"
        )

    fig, axes = plot_pca_and_umap(
        plot_df,
        color_col=preview_level_column,
        title_prefix=PRIMARY_MODEL_NAME,
    )
    # plt.savefig(plot_output_dir / f"{PRIMARY_MODEL_NAME}__{PRIMARY_METRIC}_preview_pca_umap.png", bbox_inches="tight")
    plt.show()


## Per-Model Metric Plots


In [ ]:
print("Model/metric overlays available for plotting:")
print([(model_name, metric_column) for model_name, metric_column, _, _ in model_metric_plot_columns])

for model_name, metric_column, raw_column, level_column in model_metric_plot_columns:
    fig, axes = plot_pca_and_umap(
        plot_df,
        color_col=level_column,
        title_prefix=f"{model_name} | {metric_column}",
    )
    # plt.savefig(plot_output_dir / f"{model_name}__{metric_column}_pca_umap.png", bbox_inches="tight")
    plt.show()


## Optional KMeans Exploratory Section


In [ ]:
KMEANS_N_CLUSTERS = 15


In [ ]:
if RUN_KMEANS:
    if KMEANS_N_CLUSTERS < 2:
        raise ValueError(f"KMEANS_N_CLUSTERS must be at least 2, got {KMEANS_N_CLUSTERS}.")

    kmeans = KMeans(n_clusters=KMEANS_N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
    plot_df[KMEANS_LABEL_COLUMN] = kmeans.fit_predict(X_pca)
    print(f"Added cluster labels to column {KMEANS_LABEL_COLUMN!r}.")

    fig, ax = plt.subplots(figsize=FIGSIZE)
    plot_embedding_scatter(
        plot_df,
        "umap_1",
        "umap_2",
        color_col=KMEANS_LABEL_COLUMN,
        ax=ax,
        title=f"UMAP colored by {KMEANS_LABEL_COLUMN}",
    )
    plt.tight_layout()
    # plt.savefig(plot_output_dir / f"{KMEANS_LABEL_COLUMN}_umap.png", bbox_inches="tight")
    plt.show()
else:
    print("Set RUN_KMEANS = True in the config cell to run the optional clustering section.")


## Interpretation Notes

- The PCA and UMAP coordinates are computed once from the embedding table, then reused for every model overlay.
- Each evaluation CSV is filtered to `status == "ok"` by default before joining.
- Evaluation rows are mapped back to the embeddings through `image_id`, which is converted to `sample_id` by stripping the `images/` prefix and `_image.*` suffix.
- `pixel_dice` and `instance_pq` are converted into `low` / `mid` / `high` overlays using the 25th and 75th percentiles within each model separately.
- If a model is missing rows, unmatched embedding patches are kept and colored as `Missing`.
- If you want smoother or less fragmented UMAPs, adjust the same knobs as the base notebook: `UMAP_N_NEIGHBORS`, `UMAP_MIN_DIST`, and `UMAP_INPUT_PCA_COMPONENTS`.
